# How Machines "Hear": Audio Features for Voice Interfaces

**CS474: Human Computer Interaction — Modalities: Voice Prompts**

Before a voice assistant can recognize *words*, it has to solve a simpler problem: **is anyone speaking right now?**  This is called *Voice Activity Detection* (VAD), and it is built from basic audio features you can compute with a few lines of `numpy`.

In this notebook you will:

1. Synthesize a signal that alternates between "speech-like" sound and silence (no microphone needed)
2. Compute two classic features: **short-time energy** and **zero-crossing rate**
3. Build a tiny threshold-based voice activity detector and visualize it
4. Look at a **spectrogram**, the representation real speech recognizers start from

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(474)
SR = 8000  # sample rate (samples per second)

## Part 1: Synthesize "Speech" and Silence

Vowel sounds are roughly periodic: a fundamental pitch plus harmonics.  Background silence is low-level random noise.  We'll build 3 seconds of audio: silence, a vowel-like segment, silence, another vowel at a different pitch.

In [ ]:
def vowel(duration, f0, sr=SR):
    """A crude vowel: fundamental + two harmonics, with a fade in/out."""
    t = np.arange(int(duration * sr)) / sr
    sig = (1.0 * np.sin(2 * np.pi * f0 * t)
           + 0.5 * np.sin(2 * np.pi * 2 * f0 * t)
           + 0.25 * np.sin(2 * np.pi * 3 * f0 * t))
    fade = np.minimum(1, np.minimum(t, t[::-1]) / 0.05)  # 50 ms fade
    return 0.5 * sig * fade

def silence(duration, sr=SR):
    return rng.normal(0, 0.01, int(duration * sr))  # faint background noise

audio = np.concatenate([
    silence(0.6),
    vowel(0.7, f0=140),   # lower-pitched voice
    silence(0.5),
    vowel(0.8, f0=220),   # higher-pitched voice
    silence(0.4),
])
t = np.arange(len(audio)) / SR

plt.figure(figsize=(9, 2.5))
plt.plot(t, audio, lw=0.5)
plt.xlabel('time (s)'); plt.title('Synthetic audio: silence / vowel / silence / vowel')
plt.show()

## Part 2: Short-Time Energy and Zero-Crossing Rate

We chop the signal into overlapping **frames** (~25 ms) and compute per-frame:

- **Energy** — mean of the squared samples.  Speech frames are much louder than silence.
- **Zero-crossing rate (ZCR)** — how often the waveform crosses zero.  Noisy/unvoiced sounds cross often; voiced vowels cross at roughly their pitch rate.

In [ ]:
FRAME = int(0.025 * SR)   # 25 ms
HOP = int(0.010 * SR)     # 10 ms step

starts = np.arange(0, len(audio) - FRAME, HOP)
energy = np.array([np.mean(audio[s:s+FRAME] ** 2) for s in starts])
zcr = np.array([np.mean(np.abs(np.diff(np.sign(audio[s:s+FRAME])))) / 2 for s in starts])
frame_t = (starts + FRAME / 2) / SR

fig, axes = plt.subplots(2, 1, figsize=(9, 4), sharex=True)
axes[0].plot(frame_t, energy); axes[0].set_ylabel('energy')
axes[1].plot(frame_t, zcr, color='tab:orange'); axes[1].set_ylabel('zero-crossing rate')
axes[1].set_xlabel('time (s)')
axes[0].set_title('Frame-level features')
plt.show()

## Part 3: A Threshold-Based Voice Activity Detector

The simplest VAD: a frame is "speech" if its energy exceeds a threshold.  A common trick is to set the threshold *relative to the quietest frames* (assumed to be background noise) rather than hard-coding a number — a first step toward the *automatic calibration* we discussed with eye tracking thresholds, too.

In [ ]:
noise_floor = np.percentile(energy, 20)      # estimate background level
threshold = noise_floor * 20                 # speech must be well above it
is_speech = energy > threshold

plt.figure(figsize=(9, 2.8))
plt.plot(t, audio, lw=0.5, alpha=0.6, label='audio')
plt.fill_between(frame_t, -0.9, 0.9, where=is_speech, alpha=0.25,
                 color='tab:green', label='detected speech')
plt.legend(loc='upper right'); plt.xlabel('time (s)')
plt.title('Threshold-based voice activity detection')
plt.show()

print(f"noise floor {noise_floor:.2e}, threshold {threshold:.2e}")
print(f"speech detected in {is_speech.mean()*100:.0f}% of frames")

## Part 4: The Spectrogram

A **spectrogram** shows how the frequency content changes over time.  Notice the horizontal bands (harmonics) during the vowels — and that the second vowel's bands sit higher because its pitch is higher.  Real recognizers feed representations like this into machine learning models.

In [ ]:
plt.figure(figsize=(9, 3.2))
plt.specgram(audio, NFFT=256, Fs=SR, noverlap=128, cmap='magma')
plt.xlabel('time (s)'); plt.ylabel('frequency (Hz)')
plt.title('Spectrogram')
plt.colorbar(label='power (dB)')
plt.ylim(0, 1500)
plt.show()

## Your Turn

1. **Stress test the VAD.**  Raise the background noise from `0.01` to `0.1` in `silence()` (a noisy cafe).  Does the detector still work?  Tune the multiplier `20` — what is the trade-off between *missing quiet speech* and *false triggers*?
2. **Whisper mode.**  Multiply the vowels by `0.1` to simulate whispering.  What breaks first, and what does that imply for users who cannot speak loudly?
3. **Design connection.**  In the voice prompt programming assignment, your program pauses (`pause_threshold`) to decide the user has finished talking.  Using the energy plot, explain how too short or too long a pause threshold would change the user experience — and which users each failure would harm most.

## Reflection

Voice interfaces fail differently for different people: accents, speech impairments, background environments.  A fixed threshold encodes an assumption about the "typical" user — one theme of this course is learning to notice those assumptions.